# Spark + Iceberg internal mode

Register an Iceberg INTERNAL catalog in Kasanari and verify it is visible.


In [1]:
import json
import requests

base_url = "http://kasanari:9090"
catalog_id = "iceberg_spark_internal"

payload = {
    "catalogId": catalog_id,
    "catalogType": "ICEBERG",
    "mode": "INTERNAL",
    "spec": {
        "fileIoProperties": {},
        "catalogProperties": {
            "uri": "jdbc:postgresql://catalog-storage:5432/postgres",
            "warehouse": "s3a://warehouse",
            "io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
            "s3.endpoint": "http://minio:9000",
            "s3.access-key-id": "admin",
            "s3.secret-access-key": "password",
            "s3.path-style-access": "true",
            "s3.client-factory": "kasanari.catalog.iceberg.s3.NoneRegionS3FileIOAwsClientFactory",
            "kasanari.jdbc.user": "postgres",
            "kasanari.jdbc.password": "postgres"
        }
    }
}

response = requests.post(f"{base_url}/management/v1/catalogs", json=payload, timeout=20)
print(response.status_code)
print(response.text)


201
{"catalogId":"iceberg_spark_internal","catalogType":"ICEBERG","mode":"INTERNAL","spec":{"fileIoProperties":{},"catalogProperties":{"uri":"jdbc:postgresql://catalog-storage:5432/postgres","warehouse":"s3a://warehouse","io-impl":"org.apache.iceberg.aws.s3.S3FileIO","s3.endpoint":"http://minio:9000","s3.access-key-id":"admin","s3.secret-access-key":"password","s3.path-style-access":"true","s3.client-factory":"kasanari.catalog.iceberg.s3.NoneRegionS3FileIOAwsClientFactory","kasanari.jdbc.user":"postgres","kasanari.jdbc.password":"postgres"}},"version":1}


In [2]:
response = requests.get(f"{base_url}/management/v1/catalogs/ICEBERG/{catalog_id}", timeout=20)
print(response.status_code)
print(json.dumps(response.json(), indent=2))


200
{
  "catalogId": "iceberg_spark_internal",
  "catalogType": "ICEBERG",
  "mode": "INTERNAL",
  "spec": {
    "fileIoProperties": {},
    "catalogProperties": {
      "uri": "jdbc:postgresql://catalog-storage:5432/postgres",
      "warehouse": "s3a://warehouse",
      "io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
      "s3.endpoint": "http://minio:9000",
      "s3.access-key-id": "admin",
      "s3.secret-access-key": "password",
      "s3.path-style-access": "true",
      "s3.client-factory": "kasanari.catalog.iceberg.s3.NoneRegionS3FileIOAwsClientFactory",
      "kasanari.jdbc.user": "postgres",
      "kasanari.jdbc.password": "postgres"
    }
  },
  "version": 1
}


## Spark SQL operations through Iceberg REST catalog

This section uses Spark SQL against the registered Kasanari catalog and runs a small lifecycle: create/insert/select/alter/view/delete/drop.


In [7]:
import uuid
from pyspark.sql import SparkSession

spark_catalog = "kasanari_iceberg"
base_url = "http://kasanari:9090"

spark = (
    SparkSession.builder
    .appName("kasanari-iceberg-internal-ops")
    .master("local[*]")
    .config("spark.jars", "/home/jovyan/extra-jars/iceberg-spark-runtime-4.0_2.13-1.10.1.jar")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{spark_catalog}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{spark_catalog}.type", "rest")
    .config(f"spark.sql.catalog.{spark_catalog}.uri", f"{base_url}/iceberg")
    .config(f"spark.sql.catalog.{spark_catalog}.warehouse", catalog_id)
    .config(f"spark.sql.catalog.{spark_catalog}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{spark_catalog}.s3.endpoint", "http://minio:9000")
    .config(f"spark.sql.catalog.{spark_catalog}.s3.path-style-access", "true")
    .config(f"spark.sql.catalog.{spark_catalog}.s3.access-key-id", "admin")
    .config(f"spark.sql.catalog.{spark_catalog}.s3.secret-access-key", "password")
    .getOrCreate()
)

ns = "demo"
table = f"events_{uuid.uuid4().hex[:8]}"
view = f"{table}_v"

# spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {spark_catalog}.{ns}")
# spark.sql(
#     f"""
#     CREATE TABLE IF NOT EXISTS {spark_catalog}.{ns}.{table} (
#       id INT,
#       event STRING,
#       ts TIMESTAMP,
#       source STRING
#     ) USING iceberg
#     """
# )

# spark.sql(
#     f"""
#     INSERT INTO {spark_catalog}.{ns}.{table}
#     VALUES
#       (1, "signup", current_timestamp(), "spark"),
#       (2, "click", current_timestamp(), "spark"),
#       (3, "purchase", current_timestamp(), "spark")
#     """
# )

# print("Initial rows:")
# spark.sql(f"SELECT id, event, source FROM {spark_catalog}.{ns}.{table} ORDER BY id").show(truncate=False)

# spark.sql(f"ALTER TABLE {spark_catalog}.{ns}.{table} ADD COLUMN notes STRING")
# spark.sql(f"UPDATE {spark_catalog}.{ns}.{table} SET source = 'updated-spark' WHERE id IN (1, 2)")

# print("Updated rows:")
# spark.sql(f"SELECT id, event, source FROM {spark_catalog}.{ns}.{table} ORDER BY id").show(truncate=False)

spark.sql(
    f"CREATE OR REPLACE VIEW {spark_catalog}.{ns}.{view} AS "
    f"SELECT id, event FROM {spark_catalog}.{ns}.{table} WHERE id <= 2"
)

# print("View rows:")
# spark.sql(f"SELECT * FROM {spark_catalog}.{ns}.{view} ORDER BY id").show(truncate=False)

# spark.sql(f"DELETE FROM {spark_catalog}.{ns}.{table} WHERE id = 3")

# print("After delete:")
# spark.sql(f"SELECT id, event, notes FROM {spark_catalog}.{ns}.{table} ORDER BY id").show(truncate=False)

# spark.sql(f"DROP VIEW {spark_catalog}.{ns}.{view}")
# spark.sql(f"DROP TABLE {spark_catalog}.{ns}.{table}")

# print("Done: created, inserted, selected, altered, viewed, deleted, and dropped objects.")


Py4JJavaError: An error occurred while calling o51.sql.
: java.lang.NoSuchMethodError: 'void org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier.<init>(org.apache.spark.sql.connector.catalog.CatalogPlugin, org.apache.spark.sql.connector.catalog.Identifier)'
	at org.apache.spark.sql.catalyst.analysis.RewriteViewCommands$ResolvedIdent$.unapply(RewriteViewCommands.scala:105)
	at org.apache.spark.sql.catalyst.analysis.RewriteViewCommands$$anonfun$apply$1.applyOrElse(RewriteViewCommands.scala:54)
	at org.apache.spark.sql.catalyst.analysis.RewriteViewCommands$$anonfun$apply$1.applyOrElse(RewriteViewCommands.scala:50)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsUpWithPruning$3(AnalysisHelper.scala:139)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:107)
	at org.apache.spark.sql.catalyst.plans.logical.AnalysisHelper.$anonfun$resolveOperatorsUpWithPruning$1(AnalysisHelper.scala:139)
	at org.apache.spark.sql.cataly